# Exercise 07 : Approval in a Subagent

Open `exercises/exercise07-approval-in-a-subagent.ipynb` and run the setup cell. Build a supervisor whose parts subagent pauses for approval before recording an order by replacing the `____` blanks. The solution is `solutions/exercise07-approval-in-a-subagent.ipynb`.

Worked examples to revisit : `demo01 · interrupts`, `demo02b · in-a-subagent`.

Made up, not from work. Use the supplied Aurelia data and fictional policy values.

Run the next cell first. Its code is folded : click the cell and press Shift+Enter. It loads the course key and builds `llm`, the course model.

In [1]:
#@title Course setup : run this cell first
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph-Student/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


In [2]:
import json
import operator
import sqlite3
import threading
import time
from pathlib import Path
from typing import Annotated, TypedDict
from uuid import UUID, uuid4

from langchain.agents import create_agent
from langchain.messages import HumanMessage, SystemMessage, ToolMessage
from langchain.tools import ToolRuntime, tool
from langchain_core.callbacks import BaseCallbackHandler
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.types import Command, interrupt

class Counter(BaseCallbackHandler):
    def __init__(self):
        self.lock = threading.Lock()
        self.calls = self.input_tokens = self.output_tokens = 0

    def on_chat_model_start(self, serialized, messages, **kwargs):
        with self.lock:
            self.calls += 1

    def on_llm_end(self, response, **kwargs):
        with self.lock:
            for batch in response.generations:
                for generation in batch:
                    usage = generation.message.usage_metadata or {}
                    self.input_tokens += usage.get("input_tokens", 0)
                    self.output_tokens += usage.get("output_tokens", 0)

counter = Counter()
llm = make_llm(callbacks=[counter], max_tokens=650, max_retries=0)
started = time.perf_counter()
DB = Path(data("capstone/service.db")).resolve()
LIMIT = {"recursion_limit": 24}

def part_facts(part_no):
    with sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True) as db:
        db.row_factory = sqlite3.Row
        row = db.execute("SELECT part_no, name, stock, price_eur, depot FROM parts WHERE part_no = ?",
                         (part_no,)).fetchone()
    if row is None:
        raise ValueError("Unknown Aurelia part")
    return dict(row)

def check(label, condition):
    print("check :", label, "PASS" if condition else "FAIL")
    assert condition, label

def finish_metrics(slug, server_usage=None):
    rows = server_usage or []
    result = {"model": MODEL, "rps": float(os.environ.get("COURSE_RPS", "2")),
              "calls": counter.calls + len(rows),
              "input_tokens": counter.input_tokens + sum(r.get("input_tokens", 0) for r in rows),
              "output_tokens": counter.output_tokens + sum(r.get("output_tokens", 0) for r in rows),
              "seconds": round(time.perf_counter() - started, 1)}
    print("metrics :", json.dumps(result))
    Path(slug + "-metrics.json").write_text(json.dumps(result, indent=2) + "\n", encoding="utf-8")

## Step 1 : Order policy

Every request needs a person's approval, and this step decides which person and checks that the order is valid. Rules in code are rules that a model cannot change. The cell has the three `REQUESTS` and `validated_order`, which reads the part from the database with `part_facts` and computes the total. The ledger `ORDERS` and the counters `tool_entries` and `review_entries` also start empty here.

Fill two blanks. The blank in `reviewer` is the condition for the manager : a total above 10,000 EUR, and the lead for 10,000 EUR or less. The blank in `validated_order` follows a `not`, so write the condition for a valid quantity : positive and no larger than the stock in `facts`. Module 07 demo03 · human-in-the-loop-middleware checks the same bounds.

When it worked, the cell prints an `order :` line per request with its total and reviewer, then PASS on `reviewer boundary` and `edit recomputes price`. A FAIL on `reviewer boundary` means the exact limit goes to the wrong person. A `ValueError` on a supplied request means the quantity condition is reversed.

In [3]:
ORDERS = {}
tool_entries = {}
review_entries = {}
REQUESTS = [
    {"order_id": "A-approve", "part_no": "MR-HF-01", "quantity": 5},
    {"order_id": "A-edit", "part_no": "CT-FN-02", "quantity": 3},
    {"order_id": "A-reject", "part_no": "CT-DM-04", "quantity": 1},
]

def reviewer(total_eur):
    return "manager" if total_eur > 10000 else "lead"

def validated_order(order):
    facts = part_facts(order["part_no"])
    quantity = order["quantity"]
    if type(quantity) is not int or not 0 < quantity <= facts["stock"]:
        raise ValueError("Quantity must be positive and within stock")
    total_eur = round(quantity * facts["price_eur"])
    return {**order, "total_eur": total_eur, "reviewer": reviewer(total_eur)}

for request in REQUESTS:
    print("order :", validated_order(request))
check("reviewer boundary", reviewer(10000) == "lead" and reviewer(10001) == "manager")
check("edit recomputes price", validated_order({**REQUESTS[1], "quantity": 1})["total_eur"] == 420)

order : {'order_id': 'A-approve', 'part_no': 'MR-HF-01', 'quantity': 5, 'total_eur': 1550, 'reviewer': 'lead'}
order : {'order_id': 'A-edit', 'part_no': 'CT-FN-02', 'quantity': 3, 'total_eur': 1260, 'reviewer': 'lead'}
order : {'order_id': 'A-reject', 'part_no': 'CT-DM-04', 'quantity': 1, 'total_eur': 12900, 'reviewer': 'manager'}
check : reviewer boundary PASS
check : edit recomputes price PASS


## Step 2 : Parts subagent

An approval gate needs a pause before the action and a branch per decision. The slides *The interrupt Function* and *Approve, Edit and Reject* show both, and this step builds them inside a child graph. The cell defines the `Purchase` state and the `prepare` node, which validates the order and asks the model for a short note. `review_order` counts each entry in `review_entries`, and the supplied lines after the pause refuse a decision from the wrong reviewer and record the order in `ORDERS`. `ORDERS` is a simulated ledger, written only after approval.

Fill the call that pauses the graph with the review payload, the first blank. Then replace the comment and the `____` with the decision branches on `action`. A reject returns a `result` with a rejected status and the decision's note as its reason ; the wrong-reviewer return above it shows the shape. An edit reads the new quantity from `decision`, refuses an increase, and validates the order again, which recomputes the total. Any action other than approve raises `ValueError`. An approve and a valid edit return nothing here, so they continue to the recording lines below the branches.

These cells make no model call. The first prints PASS on `child has no separate saver`. The check cell below it runs `review_order` alone on a scratch graph with a saver, so the pause and each decision run without `prepare`, and it removes its test order afterwards. When it worked, all five decision checks print PASS. A `NameError` points at the pause, and a FAIL, a `KeyError` or a `ValueError` at the branches.

In [4]:
class Purchase(TypedDict):
    order: dict
    note: str
    result: dict

def prepare(state):
    order = validated_order(state["order"])
    note = llm.invoke("Write one sentence for the Aurelia reviewer from these order facts only : "
                      "what is ordered, the total, and one thing to check before approving.\n"
                      + json.dumps(order)).content
    return {"order": order, "note": note}

def review_order(state):
    order = state["order"]
    key = order["order_id"]
    review_entries[key] = review_entries.get(key, 0) + 1
    decision = interrupt({"order": order, "note": state["note"], "options": ["approve", "edit", "reject"]})
    action = decision["action"]
    if decision.get("by") != order["reviewer"]:
        return {"result": {"order_id": key, "status": "rejected",
                           "reason": "decision needs the " + order["reviewer"]}}
    if action == "reject":
        return {"result": {"order_id": key, "status": "rejected", "reason": decision["note"]}}
    if action == "edit":
        quantity = decision["quantity"]
        if quantity > order["quantity"]:
            raise ValueError("An increased quantity needs a fresh review")
        order = validated_order({**order, "quantity": quantity})
    elif action != "approve":
        raise ValueError("Unknown approval action")
    recorded = {**order, "status": "recorded", "decision": action, "external_action": False}
    if key in ORDERS and ORDERS[key] != recorded:
        raise ValueError("Order id already recorded with different values")
    ORDERS.setdefault(key, recorded)
    return {"result": recorded}

b = StateGraph(Purchase)
b.add_node("prepare", prepare)
b.add_node("review", review_order)
b.add_edge(START, "prepare")
b.add_edge("prepare", "review")
b.add_edge("review", END)
parts_subagent = b.compile()
check("child has no separate saver", parts_subagent.checkpointer is None)

check : child has no separate saver PASS


In [5]:
def try_decision(decision):
    scratch = StateGraph(Purchase)
    scratch.add_node("review", review_order)
    scratch.add_edge(START, "review")
    scratch.add_edge("review", END)
    graph = scratch.compile(checkpointer=InMemorySaver())
    cfg = {"configurable": {"thread_id": str(uuid4())}}
    order = validated_order({"order_id": "T-check", "part_no": "CT-FN-02", "quantity": 3})
    try:
        graph.invoke({"order": order, "note": "check"}, cfg)
        result = graph.invoke(Command(resume=decision), cfg)["result"]
        return result, "T-check" in ORDERS
    finally:
        ORDERS.pop("T-check", None)
        review_entries.pop("T-check", None)

def refused(decision):
    try:
        try_decision(decision)
    except ValueError:
        return True
    return False

approved, recorded = try_decision({"action": "approve", "by": "lead"})
check("approve records the order", recorded and approved["status"] == "recorded" and approved["total_eur"] == 1260)
edited, recorded = try_decision({"action": "edit", "quantity": 1, "by": "lead"})
check("edit recomputes the total", recorded and edited["quantity"] == 1 and edited["total_eur"] == 420)
declined, recorded = try_decision({"action": "reject", "note": "Not needed.", "by": "lead"})
check("reject records nothing", not recorded and declined["status"] == "rejected" and declined["reason"] == "Not needed.")
check("raised quantity refused", refused({"action": "edit", "quantity": 5, "by": "lead"}))
check("unknown action refused", refused({"action": "hold", "by": "lead"}))

check : approve records the order PASS
check : edit recomputes the total PASS
check : reject records nothing PASS
check : raised quantity refused PASS
check : unknown action refused PASS


## Step 3 : Supervisor tool

A multi-agent approval has to pause the whole system, so the pause inside the child must reach the supervisor and the person. The slide *Interrupts in Subagents* and module 07 demo02b · in-a-subagent show a tool that invokes a child graph. The cell defines the tool `parts_specialist`, which counts each entry in `tool_entries`, and the supervisor with `create_agent`.

Fill two blanks. The first is the child graph that you compiled in Step 2. The second is the saver class from the first code cell, which goes on the supervisor only. Pass no config to the nested call : LangGraph passes the supervisor's configuration to it. Keep the tool result as JSON, and let the interrupt propagate to the supervisor instead of returning it as an ordinary tool result.

When it worked, the cell prints PASS on `supervisor saver installed` and `order ledger empty`. Python runs the body of the tool only when the model calls it, so a wrong name in the first blank shows up as a `NameError` in Step 4.

In [6]:
@tool
def parts_specialist(order_id: str, part_no: str, quantity: int) -> str:
    """Ask the parts subagent to review and record one simulated Aurelia order."""
    tool_entries[order_id] = tool_entries.get(order_id, 0) + 1
    result = parts_subagent.invoke({"order": {"order_id": order_id, "part_no": part_no, "quantity": quantity}})
    return json.dumps(result["result"])

supervisor = create_agent(llm, tools=[parts_specialist], checkpointer=InMemorySaver(),
    system_prompt="Call parts_specialist exactly once with the requested order id, part and quantity. "
                  "After its result, report the recorded or rejected status and stop. "
                  "An edited result is authoritative. Never retry a rejected order.")
check("supervisor saver installed", supervisor.checkpointer is not None)
check("order ledger empty", ORDERS == {})

check : supervisor saver installed PASS
check : order ledger empty PASS


## Step 4 : Top-level pause

You write nothing in this step : it shows the pause from the outside. The cell first empties `ORDERS` and both counters, so you can run it again, and starts each request in its own supervisor thread, with `version="v2"`, which puts the pause in `out.interrupts`, as on the slide *The interrupt Function*. `get_state(cfg).next` names the node where the supervisor waits.

Read the top-level payloads. Each one shows the order, the model's note and the options that a person can choose. The check `first entries counted` confirms that each order has one entry in both counters before any decision, and you compare that with the counters of Step 5.

A FAIL on `one nested approval` or `requested order preserved` usually means the model did not call the tool as asked, or that Step 2 or 3 has an error.

In [7]:
ORDERS.clear()
tool_entries.clear()
review_entries.clear()
configs = {}
paused = {}
for request in REQUESTS:
    key = request["order_id"]
    cfg = {"configurable": {"thread_id": str(uuid4())}, "recursion_limit": 24}
    configs[key] = cfg
    out = supervisor.invoke({"messages": [HumanMessage("Request this Aurelia order: " + json.dumps(request))]},
                            cfg, version="v2")
    paused[key] = out
    print("top-level interrupts :", [i.value for i in out.interrupts])
    print("top-level next :", supervisor.get_state(cfg).next)
    check("one nested approval", len(out.interrupts) == 1)
    check("requested order preserved", out.interrupts[0].value["order"]["order_id"] == key
          and out.interrupts[0].value["order"]["part_no"] == request["part_no"]
          and out.interrupts[0].value["order"]["quantity"] == request["quantity"])
    check("supervisor paused in tools", supervisor.get_state(cfg).next == ("tools",))
check("nothing recorded before approval", not ORDERS)
check("first entries counted", all(tool_entries[r["order_id"]] == review_entries[r["order_id"]] == 1 for r in REQUESTS))

top-level interrupts : [{'order': {'order_id': 'A-approve', 'part_no': 'MR-HF-01', 'quantity': 5, 'total_eur': 1550, 'reviewer': 'lead'}, 'note': 'Order A-approve is for 5 units of MR-HF-01, totaling €1,550; check that the quantity matches the order before approving.', 'options': ['approve', 'edit', 'reject']}]
top-level next : ('tools',)
check : one nested approval PASS
check : requested order preserved PASS
check : supervisor paused in tools PASS


top-level interrupts : [{'order': {'order_id': 'A-edit', 'part_no': 'CT-FN-02', 'quantity': 3, 'total_eur': 1260, 'reviewer': 'lead'}, 'note': 'The order is for 3 units of part CT-FN-02, totaling €1,260; check the part number before approving.', 'options': ['approve', 'edit', 'reject']}]
top-level next : ('tools',)
check : one nested approval PASS
check : requested order preserved PASS
check : supervisor paused in tools PASS


top-level interrupts : [{'order': {'order_id': 'A-reject', 'part_no': 'CT-DM-04', 'quantity': 1, 'total_eur': 12900, 'reviewer': 'manager'}, 'note': 'One CT-DM-04 is ordered for €12,900 total; check that the quantity and total match the order before approving.', 'options': ['approve', 'edit', 'reject']}]
top-level next : ('tools',)
check : one nested approval PASS
check : requested order preserved PASS
check : supervisor paused in tools PASS
check : nothing recorded before approval PASS
check : first entries counted PASS


## Step 5 : Decisions

A paused thread continues when a person's decision arrives, as on the slide *Resuming with Command*. The slide also explains the counters : the paused node runs again from its first line, so code before `interrupt` runs a second time. The cell has one decision per order and a loop that resumes each saved supervisor thread. Each decision names who took it ; `review_order` refuses a decision from the wrong reviewer.

Fill the action of each decision with the option names that Step 4 printed, as strings in quotes : the lead approves the helium filters and reduces the fan order to one unit, and the manager rejects the detector module. Then fill the `Command` blank in the loop with the decision of the current pass.

When it worked, each order prints a `parts result :` line and a `supervisor :` line, then `entry counts :` and `simulated orders :`. Compare the simulated ledger with both counters. The completed preparation node, including its model call, stays checkpointed ; the paused tool wrapper and review node execute again.

A `KeyError` on `ORDERS`, or a FAIL on `edited quantity and price` or `rejection keeps the reason`, points at the branches of Step 2. Each thread takes one decision, so after a fix run the cells from Step 1 to here in order. Then run the metrics cell at the end of the notebook, under *If time permits*.

In [8]:
decisions = {
    "A-approve": {"action": "approve", "by": "lead"},
    "A-edit": {"action": "edit", "quantity": 1, "by": "lead"},
    "A-reject": {"action": "reject", "note": "Repair the detector first. Manager declines the purchase.", "by": "manager"},
}
completed = {}
for key, decision in decisions.items():
    out = supervisor.invoke(Command(resume=decision), configs[key], version="v2")
    completed[key] = out
    print("decision :", key, decision)
    for message in out.value["messages"]:
        if isinstance(message, ToolMessage):
            print("parts result :", message.content)
    print("supervisor :", out.value["messages"][-1].content)
    check("completed without another interrupt", not out.interrupts)
check("approved quantity retained", ORDERS["A-approve"]["quantity"] == 5 and ORDERS["A-approve"]["total_eur"] == 1550)
check("edited quantity and price", ORDERS["A-edit"]["quantity"] == 1 and ORDERS["A-edit"]["total_eur"] == 420)
rejected_results = [json.loads(m.content) for m in completed["A-reject"].value["messages"] if isinstance(m, ToolMessage)]
check("rejection recorded without order", "A-reject" not in ORDERS and rejected_results[-1]["status"] == "rejected")
check("rejection keeps the reason", rejected_results[-1]["reason"] == decisions["A-reject"]["note"])
print("entry counts :", {"tool": tool_entries, "review": review_entries})
print("simulated orders :", ORDERS)
check("code before interrupt replayed", all(tool_entries[r["order_id"]] == review_entries[r["order_id"]] == 2 for r in REQUESTS))
check("each approved action recorded once", len(ORDERS) == 2 and all(not o["external_action"] for o in ORDERS.values()))

decision : A-approve {'action': 'approve', 'by': 'lead'}
parts result : {"order_id": "A-approve", "part_no": "MR-HF-01", "quantity": 5, "total_eur": 1550, "reviewer": "lead", "status": "recorded", "decision": "approve", "external_action": false}
supervisor : Recorded: order A-approve for 5 × MR-HF-01.
check : completed without another interrupt PASS


decision : A-edit {'action': 'edit', 'quantity': 1, 'by': 'lead'}
parts result : {"order_id": "A-edit", "part_no": "CT-FN-02", "quantity": 1, "total_eur": 420, "reviewer": "lead", "status": "recorded", "decision": "edit", "external_action": false}
supervisor : Recorded with the edited quantity of 1.
check : completed without another interrupt PASS


decision : A-reject {'action': 'reject', 'note': 'Repair the detector first. Manager declines the purchase.', 'by': 'manager'}
parts result : {"order_id": "A-reject", "status": "rejected", "reason": "Repair the detector first. Manager declines the purchase."}
supervisor : Order A-reject was rejected. The detector must be repaired first; the manager declined the purchase.
check : completed without another interrupt PASS
check : approved quantity retained PASS
check : edited quantity and price PASS
check : rejection recorded without order PASS
check : rejection keeps the reason PASS
entry counts : {'tool': {'A-approve': 2, 'A-edit': 2, 'A-reject': 2}, 'review': {'A-approve': 2, 'A-edit': 2, 'A-reject': 2}}
simulated orders : {'A-approve': {'order_id': 'A-approve', 'part_no': 'MR-HF-01', 'quantity': 5, 'total_eur': 1550, 'reviewer': 'lead', 'status': 'recorded', 'decision': 'approve', 'external_action': False}, 'A-edit': {'order_id': 'A-edit', 'part_no': 'CT-FN-02', 'quantity': 1, 'total_

### If time permits

- Edit an order to quantity 0 or above stock and read the validation error
- Print the supervisor state history and find the checkpoint with the pending approval
- Add a second review by the manager after the lead approves an expensive order
- Resume the detector order with `"by": "lead"` in a fresh thread and read the result

In [9]:
finish_metrics("exercise07-approval-in-a-subagent")

metrics : {"model": "openai/gpt-6-luna", "rps": 0.9, "calls": 9, "input_tokens": 1353, "output_tokens": 260, "seconds": 12.7}
